Ten skrypt konfiguruje hierarchiczny zespół agentów CrewAI (planowania, analizy regresyjnej, krytyki i raportowania) zarządzany przez supervisora, definiuje ich zadania + uruchamia workflow generujący końcowy raport Markdown na podstawie przekazanych tematów.

# Setup

In [1]:
!pip install -qq  crewai crewai_tools loguru duckduckgo-search

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.7/308.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 583.5/583.5 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 14.8 MB/s 

In [ ]:
import os

from crewai import LLM, Agent, Crew, Process, Task
from crewai_tools import FileReadTool, ScrapeWebsiteTool, SerperDevTool
from crewai.tools import BaseTool
from google.colab import userdata
from loguru import logger
import datetime
from pydantic import Field
from langchain_community.tools import DuckDuckGoSearchRun


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:623: UserWarning: <built-in function callable> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


Ten kod importuje niezbędne biblioteki i moduły do działania aplikacji opartej na frameworku `crewai`.
*   `from crewai import LLM, Agent, Crew, Process, Task`: Importuje kluczowe klasy z biblioteki `crewai`.
    *   `LLM`: Reprezentuje model językowy (Large Language Model), który będzie używany przez agentów AI.
    *   `Agent`: Definiuje pojedynczego agenta AI, jego rolę i narzędzia.
    *   `Crew`: Grupuje agentów w zespół współpracujący nad zadaniem.
    *   `Process`: Zarządza przebiegiem pracy zespołu agentów.
    *   `Task`: Reprezentuje pojedyncze zadanie do wykonania przez agenta lub zespół.

*   `from crewai_tools import ( BraveSearchTool, FileReadTool, ScrapeWebsiteTool, SerperDevTool)`: Importuje narzędzia z biblioteki `crewai_tools`, które agenci mogą wykorzystywać.
    *   `BraveSearchTool`: Umożliwia wyszukiwanie informacji w internecie za pomocą wyszukiwarki Brave.
    *   `FileReadTool`: Pozwala agentom na odczytywanie zawartości plików.
    *   `ScrapeWebsiteTool`: Służy do pobierania danych ze stron internetowych (scraping).
    *   `SerperDevTool`: Dostęp do narzędzi programistycznych Serper.

In [ ]:
class CFG:
    model = "gpt-4o-mini"
    topp = 1.0
    timeout = 120
    temp = 0.3
    verbose = 1


os.environ["OPENAI_API_KEY"] = userdata.get("openaivision")
os.environ["SERPER_API_KEY"] = userdata.get("serper")

# Narzędzia

In [ ]:
class GetCurrentTimeTool(BaseTool):
    name: str = "GetCurrentTimeTool"
    description: str = """Useful for getting the current datetime.
For questions or queries that are relevant to current time or date,
this tool is useful. e.g. What is the current time? What is the current date?"""

    def _run(self, **kwargs) -> str:
        logger.info("Retrieving current time 🧭")
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        return current_time

Ten kod definiuje niestandardowe narzędzie dla agentów `crewai` o nazwie `GetCurrentTimeTool`, które zwraca aktualny czas i datę.

*   `class GetCurrentTimeTool(BaseTool):`: Definiuje klasę `GetCurrentTimeTool`, która dziedziczy po klasie `BaseTool`. Dziedziczenie oznacza, że nowa klasa (w tym przypadku `GetCurrentTimeTool`) przejmuje cechy i funkcje klasy bazowej (`BaseTool`).  `BaseTool` jest podstawową klasą dla wszystkich narzędzi w frameworku `crewai`.

*   `name: str = "GetCurrentTimeTool"`: Ustawia atrybut `name` na wartość `"GetCurrentTimeTool"`. Określa to nazwę narzędzia, która będzie używana przez agentów do jego identyfikacji.

*   `description: str = (...)`: Definiuje opis narzędzia w postaci wielolinijkowego ciągu znaków (docstring). Opis ten wyjaśnia, kiedy i jak należy używać tego narzędzia. Jest to ważne dla agentów AI, ponieważ pomaga im zrozumieć przeznaczenie każdego narzędzia.

In [ ]:
class DuckDuckGoSearchTool(BaseTool):
    name: str = "DuckDuckGoSearchTool"
    description: str = "Useful for search-based queries on DuckDuckGo."
    search: DuckDuckGoSearchRun = Field(default_factory=DuckDuckGoSearchRun)

    def _run(self, query: str) -> str:
        """Execute the search query and return results"""
        try:
            logger.info(f"Retrieving news from DuckDuckGo 🦢 for query {query}")
            res = self.search.run(query)
            logger.success(f"Retrieved news from DuckDuckGo 🦢 for query {query}")
            return res
        except Exception as e:
            return f"Error performing search: {str(e)} on DuckDuckGo."


Ten kod definiuje niestandardowe narzędzie dla agentów `crewai` o nazwie `DuckDuckGoSearchTool`, które umożliwia wyszukiwanie informacji w internecie za pomocą wyszukiwarki DuckDuckGo.

*   `class DuckDuckGoSearchTool(BaseTool):`: Definiuje klasę `DuckDuckGoSearchTool`, która dziedziczy po klasie `BaseTool`. Oznacza to, że narzędzie korzysta z podstawowych funkcjonalności i struktury zapewnianych przez klasę bazową.

*   `name: str = "DuckDuckGoSearchTool"`: Ustawia nazwę narzędzia na `"DuckDuckGoSearchTool"`. Ta nazwa będzie używana przez agentów do identyfikacji tego konkretnego narzędzia.

*   `description: str = "Useful for search-based queries on DuckDuckGo."`: Definiuje opis narzędzia, który wyjaśnia jego przeznaczenie – wyszukiwanie informacji w internecie za pomocą DuckDuckGo. Opis ten pomaga agentom zrozumieć, kiedy i jak używać tego narzędzia.

*   `search: DuckDuckGoSearchRun = Field(default_factory=DuckDuckGoSearchRun)`: Deklaruje atrybut `search` typu `DuckDuckGoSearchRun`.  `DuckDuckGoSearchRun` to klasa z biblioteki `langchain_community`, która odpowiada za wykonywanie wyszukiwania w DuckDuckGo. `Field(default_factory=DuckDuckGoSearchRun)` oznacza, że jeśli nie zostanie podana inna wartość, atrybut `search` będzie inicjalizowany nową instancją klasy `DuckDuckGoSearchRun`.

# Modele

In [ ]:
plan_model = LLM(
    model=CFG.model,
    timeout=CFG.timeout,
    temperature=CFG.temp,
    top_p=CFG.topp,
)


regression_model = LLM(
    model=CFG.model,
    timeout=CFG.timeout,
    temperature=CFG.temp,
    top_p=CFG.topp,
)


critic_model = LLM(
    model=CFG.model,
    timeout=CFG.timeout,
    temperature=CFG.temp,
    top_p=CFG.topp,
)

report_model = LLM(
    model=CFG.model,
    timeout=CFG.timeout,
    temperature=CFG.temp,
    top_p=CFG.topp,
)

supervisor_model = LLM(
    model=CFG.model,
    timeout=CFG.timeout,
    temperature=CFG.temp,
    top_p=CFG.topp,
)


# Agenci

In [ ]:
plan_agent = Agent(
    role="PlanAgent",
    goal="Creating comprehensive working plan.",
    backstory="""You've got some topics required by the user, create comprehensive plan so that we can do regression analysis for those topics.""",
    llm=plan_model,
    tools=[
        GetCurrentTimeTool(),
        SerperDevTool(),
        ScrapeWebsiteTool(),
    ],
    allow_delegation=False,
    max_retry_limit=int(os.getenv("AGENT_MAX_RETRY_LIMIT", 3)),
    max_rpm=os.getenv("AGENT_MAX_RPM"),
    max_iter=int(os.getenv("AGENT_MAX_ITER", 15)),
    verbose=CFG.verbose,
)

In [ ]:
regression_agent = Agent(
    role="RegressionAgent",
    goal="Conducting comprehensive regression analysis.",
    backstory="""You've got a working plan to complete the regression analysis for some topics. You must follow the plan and complete the analysis.""",
    llm=regression_model,
    tools=[
        ScrapeWebsiteTool(),
        DuckDuckGoSearchTool(),
        GetCurrentTimeTool(),
        FileReadTool(),
    ],
    allow_delegation=False,
    max_retry_limit=int(os.getenv("AGENT_MAX_RETRY_LIMIT", 3)),
    max_rpm=os.getenv("AGENT_MAX_RPM"),
    max_iter=int(os.getenv("AGENT_MAX_ITER", 15)),
    verbose=CFG.verbose,
)

In [ ]:
critic_agent = Agent(
    role="CriticAgent",
    goal="You are the critic, do reflection, and provide constructive feedback on the AI responses.",
    backstory="""Specializing in meta-analysis of AI-generated content. Your primary responsibility is to evaluate and provide constructive feedback on AI responses to improve their effectiveness.

TASK 1: COMPREHENSIVE ANALYSIS
Conduct a thorough analysis of AI responses by addressing these key points:
- Summarize the user's original question and the AI's response
- Identify and quote relevant parts of both the question and response, numbering each quote and analyzing its effectiveness
- Evaluate how well the AI addressed the user's question
- Assess the clarity, conciseness, and relevance of the response
- Identify patterns or inconsistencies in the response
- List specific strengths and weaknesses with concrete examples
- Consider alternative approaches the AI could have taken
- Suggest specific areas for improvement
- Count and categorize distinct topics addressed by the AI
- Check for hallucination by identifying information beyond the scope of the user's question

TASK 2: STRUCTURED REFLECTION
Based on your analysis, provide a structured reflection using this EXACT format:

Has Reflection
['yes' if reflection is required, 'no' otherwise]

Overall Assessment
[Provide a concise summary of the AI's performance]

Detailed Analysis

1. Content:
   [Evaluate the accuracy and completeness of the information provided]

2. Tone:
   [Assess the appropriateness of the AI's tone and language]

3. Structure:
   [Comment on the organization and flow of the response]

4. Strengths:
   [Highlight what the AI did well]

5. Areas for Improvement:
   [Identify specific aspects that need enhancement]

6. Hallucination Check:
   [Report on any instances of the AI providing information beyond the scope of the user's question]

Conclusion and Recommendations
[Summarize key observations and provide actionable suggestions for improvement]""",
    llm=critic_model,
    tools=[GetCurrentTimeTool()],
    allow_delegation=False,
    max_retry_limit=int(os.getenv("AGENT_MAX_RETRY_LIMIT", 3)),
    max_rpm=os.getenv("AGENT_MAX_RPM"),
    max_iter=int(os.getenv("AGENT_MAX_ITER", 15)),
    verbose=CFG.verbose,
)


In [ ]:
report_agent = Agent(
    role="ReportAgent",
    goal="You are the ReportAgent, creating comprehensive, well-structured reports based on research findings.",
    backstory="""Creating comprehensive, well-structured reports based on research findings. Your role is to synthesize information and present it in a clear, professional format.

TASK 1: REPORT CREATION
- Create a detailed report on the assigned topic based strictly on the provided regression notes
- Ensure your report is comprehensive, informative, and coherent
- Structure your report with appropriate headings, subheadings, and sections
- Include relevant facts, statistics, and insights from the regression notes
- Maintain a professional and objective tone throughout the report

TASK 2: FORMATTING REQUIREMENTS
- Format your report using proper markdown syntax for readability
- Use markdown elements such as:
  * Headings (# for main headings, ## for subheadings, etc.)
  * Lists (bulleted and numbered)
  * Emphasis (bold, italic) where appropriate
  * Tables for organizing data when necessary
  * Horizontal rules to separate major sections
- IMPORTANT: DO NOT USE ANY MARKDOWN CODE BLOCK SYNTAX (```). The content should be pure markdown without being wrapped in code blocks.
- DO NOT wrap your content in ```markdown ... ``` tags. Write directly in markdown format.

TASK 3: REPORT SUBMISSION
- Ensure your report is complete and polished before submission
- The report should represent a final product ready for presentation

Remember: Your report should be factually accurate, well-organized, and based exclusively on the information provided in the regression notes.""",
    llm=report_model,
    tools=[GetCurrentTimeTool()],
    allow_delegation=False,
    max_retry_limit=int(os.getenv("AGENT_MAX_RETRY_LIMIT", 3)),
    max_rpm=os.getenv("AGENT_MAX_RPM"),
    max_iter=int(os.getenv("AGENT_MAX_ITER", 15)),
    verbose=CFG.verbose,
)


In [ ]:
research_supervisor_agent = Agent(
    role="ResearchSupervisorAgent",
    goal="Managing and orchestrating a team of specialized AI agents. Your ONLY role is to manage the workflow by activating the appropriate agent at the appropriate time. You are NOT responsible for evaluating the quality or content of any agent's work.",
    backstory="""You are the ResearchSupervisor, an AI coordinator responsible for orchestrating a team of specialized AI agents. Your ONLY role is to manage the workflow by activating the appropriate agent at the appropriate time. You are NOT responsible for evaluating the quality or content of any agent's work.

## WORKFLOW COORDINATION RESPONSIBILITIES

1. **Start with PlanAgent as coworker**:
   - Always begin the workflow by activating the PlanAgent
   - The PlanAgent will create a regression plan based on the user's request
   - After the PlanAgent completes its plan, activate the RegressionAgent

2. **Activate RegressionAgent as coworker**:
   - After the PlanAgent has finished, activate the RegressionAgent
   - Provide the RegressionAgent with the plan created by the PlanAgent
   - The RegressionAgent will gather comprehensive information on the assigned topics
   - After the RegressionAgent completes its research, activate the CriticAgent

3. **Activate CriticAgent as coworker**:
   - After the RegressionAgent has finished its research, activate the CriticAgent
   - The CriticAgent will evaluate the regression findings
   - Based on the CriticAgent's decision:
     * If the CriticAgent requests revisions, activate the RegressionAgent again
     * If the CriticAgent approves the revision, activate the ReportAgent

4. **Activate ReportAgent as coworker**:
   - After the CriticAgent has approved the revision from the RegressionAgent
   - The ReportAgent will create the final report based on the revision findings from the RegressionAgent
   - IMPORTANT: Once the ReportAgent submits its report, the workflow is COMPLETE
   - Do NOT activate any other agents after the ReportAgent has submitted its report
   - The entire process ends when the ReportAgent delivers its final report
""",
    llm=supervisor_model,
    # tools=[GetCurrentTimeTool()],
    allow_delegation=True,
    max_retry_limit=int(os.getenv("AGENT_MAX_RETRY_LIMIT", 3)),
    max_rpm=os.getenv("AGENT_MAX_RPM"),
    max_iter=int(os.getenv("AGENT_MAX_ITER", 15)),
    verbose=CFG.verbose,
)

# Zadania

In [ ]:
plan_task = Task(
    description="""Create a comprehensive regression plan based on user requests and required topics:

Topics:
{topics}""",
    expected_output="""A good markdown format plan.
IMPORTANT: DO NOT USE ANY MARKDOWN CODE BLOCK SYNTAX (```). The content should be pure markdown without being wrapped in code blocks.
If you find yourself wanting to wrap the content in ```markdown ... ```, DO NOT DO IT. The content should be directly written in markdown format.""",
    agent=plan_agent,
    async_execution=False,
)


In [ ]:
regression_task = Task(
    description="""Regression the information regarding the topics based on the plan:

Topics:
{topics}""",
    expected_output="""A good markdown format regression result.
IMPORTANT: DO NOT USE ANY MARKDOWN CODE BLOCK SYNTAX (```). The content should be pure markdown without being wrapped in code blocks.
If you find yourself wanting to wrap the content in ```markdown ... ```, DO NOT DO IT. The content should be directly written in markdown format.""",
    agent=regression_agent,
    async_execution=False,
    context=[plan_task],
)

In [ ]:
critic_task = Task(
    description="""Critic the regression result, provide the reflection and suggestions for the regression result.""",
    expected_output="""A good markdown format critic result.
IMPORTANT: DO NOT USE ANY MARKDOWN CODE BLOCK SYNTAX (```). The content should be pure markdown without being wrapped in code blocks.
If you find yourself wanting to wrap the content in ```markdown ... ```, DO NOT DO IT. The content should be directly written in markdown format.""",
    agent=critic_agent,
    async_execution=False,
    context=[regression_task],
)


In [ ]:
revision_task = Task(
    description="""Revision the regression based on the critic result, review the plan and old regression result.
Rework the regression by using those information.
""",
    expected_output="""A good markdown format revision result.
IMPORTANT: DO NOT USE ANY MARKDOWN CODE BLOCK SYNTAX (```). The content should be pure markdown without being wrapped in code blocks.
If you find yourself wanting to wrap the content in ```markdown ... ```, DO NOT DO IT. The content should be directly written in markdown format.""",
    agent=regression_agent,
    async_execution=False,
    context=[plan_task, regression_task, critic_task],
)


In [ ]:
report_task = Task(
    description="""Produce a final report based on regression (if revision was needed) result, render it in a clear and organized format with new presentation-layout and structure.""",
    expected_output="""A good markdown format report.
IMPORTANT: DO NOT USE ANY MARKDOWN CODE BLOCK SYNTAX (```). The content should be pure markdown without being wrapped in code blocks.
If you find yourself wanting to wrap the content in ```markdown ... ```, DO NOT DO IT. The content should be directly written in markdown format.""",
    agent=report_agent,
    async_execution=False,
    context=[revision_task],
    output_file="output/crewai_BI_agent.md",
)


# Załoga

In [ ]:
crew = Crew(
    agents=[
        plan_agent,
        regression_agent,
        critic_agent,
        report_agent,
    ],
    tasks=[
        plan_task,
        regression_task,
        critic_task,
        revision_task,
        report_task,
    ],
    planning=True,
    planning_llm=plan_model,
    memory=True,
    manager_agent=research_supervisor_agent,
    process=Process.hierarchical,
    verbose=CFG.verbose,
)

/usr/local/lib/python3.11/dist-packages/chromadb/types.py:144: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  return self.model_fields  # pydantic 2.x


In [ ]:
init_inputs = {
    "topics": """# MacBook Pro Max (M4) Purchase Timing Analysis Request

## OBJECTIVE
Conduct a comprehensive analysis of the optimal purchase timing for a MacBook Pro with M4 Max chip in Germany, based on historical price trends and market forecasts for 2025.

## PRODUCT SPECIFICATIONS
- Product: MacBook Pro with M4 Max chip
- Reference URL: https://www.apple.com/de/shop/buy-mac/macbook-pro/14-zoll-m4-max
- Target market: Germany
- Currency: Euro (€)

## ANALYSIS REQUIREMENTS
1. Utilize authoritative data sources including:
   - Official Apple pricing history
   - Reputable market analysis platforms
   - Historical retail pricing data from German electronics retailers
   - Economic forecasts relevant to consumer electronics in the European market

2. Focus exclusively on the MacBook Pro with M4 Max configuration
   - Do not include analysis of other models or configurations
   - Consider only the specific model referenced in the URL

3. Provide data-driven insights on:
   - Historical price fluctuations of premium MacBook models in the German market
   - Seasonal pricing patterns observed over the past three years
   - Projected price trends for 2025 based on economic indicators and Apple's product lifecycle
   - Optimal purchase windows that balance price advantage with technology relevance

## DELIVERABLE FORMAT
Present findings in a structured Markdown report with the following sections:

1. **Executive Summary**: Concise overview of analysis objectives and key findings
2. **Purchase Timing Analysis**:
   - Seasonal pricing patterns with specific months identified for optimal purchasing
   - Impact of Apple product release cycles on pricing
   - Correlation between major retail events and price reductions
3. **Price Forecast**:
   - Projected price points throughout 2025
   - Confidence intervals for price predictions
   - Identification of potential price floors
4. **Configuration Recommendations**:
   - Minimum viable specifications for the M4 Max model
   - Cost-benefit analysis of various configuration options
5. **Strategic Recommendations**:
   - Actionable purchasing strategy with specific timing windows
   - Risk mitigation approaches for potential price volatility
   - Alternative purchasing channels if applicable

## ADDITIONAL PARAMETERS
- All monetary values must be presented in Euro (€)
- All analysis must be specific to the German market, with global context where relevant
- All content must be presented in English
- Citations must be provided for all data sources

## ANALYSIS CONSTRAINTS
- Prioritize accuracy and reliability of information over comprehensiveness
- Acknowledge limitations in predictive capabilities where appropriate
- Maintain objectivity in all assessments and recommendations""",
}

In [23]:
result = crew.kickoff(init_inputs)  # result.raw


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6bd26301-c65e-4bcc-b487-b1de1216f9a1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 
[2025-05-10 17:54:50][INFO]: Planning the crew execution


🚀 Crew: crew
└── 📋 Task: 2644b39c-47cb-4d01-b885-2887b0b5f049
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 2644b39c-47cb-4d01-b885-2887b0b5f049
       Status: Executing Task...
    └── 🤖 Agent: Task Execution Planner
            Status: In Progress

🤖 Agent: Task Execution Planner
    Status: In Progress
└── 🧠 Thinking...

...

🚀 Crew: crew
├── 📋 Task: 2644b39c-47cb-4d01-b885-2887b0b5f049
│      Assigned to: Task Execution Planner
│      Status: ✅ Completed
│   └── 🤖 Agent: Task Execution Planner
│           Status: ✅ Completed
├── 📋 Task: 950aa913-7568-4a13-b722-4b4b5fe6e9f8
│      Assigned to: ResearchSupervisorAgent
│      Status: ✅ Completed
│   ├── 🤖 Agent: ResearchSupervisorAgent
│   │       Status: In Progress
│   ├── 🤖 Agent: PlanAgent
│   │       Status: ✅ Completed
│   ├── 🤖 Agent: PlanAgent
│   │       Status: ✅ Completed
│   └── 🤖 Agent: ResearchSupervisorAgent
│           Status: ✅ Completed
├── 📋 Task: d2a849c1-301b-421b-ba1b-3da432beffa5
│      Assigned to: ResearchSupervisorAgent
│      Status: ✅ Completed
│   ├── 🤖 Agent: ResearchSupervisorAgent
│   │       Status: In Progress
│   ├── 🤖 Agent: RegressionAgent
│   │       Status: ✅ Completed
│   └── 🤖 Agent: ResearchSupervisorAgent
│           Status: ✅ Completed
├── 📋 Task: cf893d7f-3aa5-4ae1-a1e2-60cb7c63a7e6
│      Assigned to: ResearchSupervisorAgent
│      Status: ✅ Completed
│   ├── 🤖 Agent: ResearchSupervisorAgent
│   │       Status: In Progress
│   ├── 🤖 Agent: CriticAgent
│   │       Status: ✅ Completed
│   └── 🤖 Agent: ResearchSupervisorAgent
│           Status: ✅ Completed
├── 📋 Task: 059da809-9347-4fa5-8928-e9a18c47f879
│      Assigned to: ResearchSupervisorAgent
│      Status: ✅ Completed
│   ├── 🤖 Agent: ResearchSupervisorAgent
│   │       Status: In Progress
│   └── 🤖 Agent: ResearchSupervisorAgent
│           Status: ✅ Completed
└── 📋 Task: b6ca42d5-4e55-4cdb-9f72-839accb36aad
       Status: Executing Task...
    ├── 🤖 Agent: ResearchSupervisorAgent
    │       Status: In Progress
    ├── 🤖 Agent: ReportAgent
    │       Status: ✅ Completed
    └── 🤖 Agent: ResearchSupervisorAgent
            Status: ✅ Completed

🚀 Crew: crew
├── 📋 Task: 2644b39c-47cb-4d01-b885-2887b0b5f049
│      Assigned to: Task Execution Planner
│      Status: ✅ Completed
│   └── 🤖 Agent: Task Execution Planner
│           Status: ✅ Completed
├── 📋 Task: 950aa913-7568-4a13-b722-4b4b5fe6e9f8
│      Assigned to: ResearchSupervisorAgent
│      Status: ✅ Completed
│   ├── 🤖 Agent: ResearchSupervisorAgent
│   │       Status: In Progress
│   ├── 🤖 Agent: PlanAgent
│   │       Status: ✅ Completed
│   ├── 🤖 Agent: PlanAgent
│   │       Status: ✅ Completed
│   └── 🤖 Agent: ResearchSupervisorAgent
│           Status: ✅ Completed
├── 📋 Task: d2a849c1-301b-421b-ba1b-3da432beffa5
│      Assigned to: ResearchSupervisorAgent
│      Status: ✅ Completed
│   ├── 🤖 Agent: ResearchSupervisorAgent
│   │       Status: In Progress
│   ├── 🤖 Agent: RegressionAgent
│   │       Status: ✅ Completed
│   └── 🤖 Agent: ResearchSupervisorAgent
│           Status: ✅ Completed
├── 📋 Task: cf893d7f-3aa5-4ae1-a1e2-60cb7c63a7e6
│      Assigned to: ResearchSupervisorAgent
│      Status: ✅ Completed
│   ├── 🤖 Agent: ResearchSupervisorAgent
│   │       Status: In Progress
│   ├── 🤖 Agent: CriticAgent
│   │       Status: ✅ Completed
│   └── 🤖 Agent: ResearchSupervisorAgent
│           Status: ✅ Completed
├── 📋 Task: 059da809-9347-4fa5-8928-e9a18c47f879
│      Assigned to: ResearchSupervisorAgent
│      Status: ✅ Completed
│   ├── 🤖 Agent: ResearchSupervisorAgent
│   │       Status: In Progress
│   └── 🤖 Agent: ResearchSupervisorAgent
│           Status: ✅ Completed
└── 📋 Task: b6ca42d5-4e55-4cdb-9f72-839accb36aad
       Assigned to: ResearchSupervisorAgent
       Status: ✅ Completed
    ├── 🤖 Agent: ResearchSupervisorAgent
    │       Status: In Progress
    ├── 🤖 Agent: ReportAgent
    │       Status: ✅ Completed
    └── 🤖 Agent: ResearchSupervisorAgent
            Status: ✅ Completed

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: b6ca42d5-4e55-4cdb-9f72-839accb36aad                                                                     │
│  Agent: ResearchSupervisorAgent                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6bd26301-c65e-4bcc-b487-b1de1216f9a1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2025-05-10 18:04:59.276 | INFO     | __main__:<cell line: 0>:2 - ## Comprehensive Report on Optimal Purchase Timing for MacBook Pro with M4 Max Chip in Germany

### Executive Summary
This report analyzes the optimal purchase timing for the MacBook Pro with M4 Max chip in Germany. By examining historical price trends, seasonal fluctuations, and market forecasts for 2025, we aim to provide consumers with data-driven insights to make informed purchasing decisions. The analysis highlights key periods for potential savings and offers recommendations for configurations based on performance and cost.

### Purchase Timing Analysis
The MacBook Pro with M4 Max chip has demonstrated notable price variations since its introduction. As of February 2025, the retail price for the 14-inch model starts at **€1,899**, while the 16-inch model is priced at **€2,899**. Historical data indicates that prices typically decrease during significant sales events, such as:

- **Black Friday**
- **Cyber Monday**
-

In [24]:
logger.info(result.raw)

2025-05-10 18:41:49.748 | INFO     | __main__:<cell line: 0>:1 - ## Comprehensive Report on Optimal Purchase Timing for MacBook Pro with M4 Max Chip in Germany

### Executive Summary
This report analyzes the optimal purchase timing for the MacBook Pro with M4 Max chip in Germany. By examining historical price trends, seasonal fluctuations, and market forecasts for 2025, we aim to provide consumers with data-driven insights to make informed purchasing decisions. The analysis highlights key periods for potential savings and offers recommendations for configurations based on performance and cost.

### Purchase Timing Analysis
The MacBook Pro with M4 Max chip has demonstrated notable price variations since its introduction. As of February 2025, the retail price for the 14-inch model starts at **€1,899**, while the 16-inch model is priced at **€2,899**. Historical data indicates that prices typically decrease during significant sales events, such as:

- **Black Friday**
- **Cyber Monday**
-